In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [4]:
#train.shape
#train.head()
train.isna().mean().sort_values(ascending=False).head(20)

PoolQC          0.995205
MiscFeature     0.963014
Alley           0.937671
Fence           0.807534
MasVnrType      0.597260
FireplaceQu     0.472603
LotFrontage     0.177397
GarageQual      0.055479
GarageFinish    0.055479
GarageType      0.055479
GarageYrBlt     0.055479
GarageCond      0.055479
BsmtFinType2    0.026027
BsmtExposure    0.026027
BsmtCond        0.025342
BsmtQual        0.025342
BsmtFinType1    0.025342
MasVnrArea      0.005479
Electrical      0.000685
Condition2      0.000000
dtype: float64

In [5]:
train["SalePrice"].describe()

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

In [6]:
train["SalePrice"].skew()

np.float64(1.8828757597682129)

In [7]:
train[[
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "YearBuilt",
    "FullBath",
    "TotRmsAbvGrd",
    "SalePrice"
]].corr()["SalePrice"].sort_values(ascending=False)

SalePrice       1.000000
OverallQual     0.790982
GrLivArea       0.708624
GarageCars      0.640409
GarageArea      0.623431
TotalBsmtSF     0.613581
1stFlrSF        0.605852
FullBath        0.560664
TotRmsAbvGrd    0.533723
YearBuilt       0.522897
Name: SalePrice, dtype: float64

In [8]:
train.groupby("OverallQual")["SalePrice"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
OverallQual,,,
1,2,50150.000000,50150.0
2,3,51770.333333,60000.0
3,20,87473.750000,86250.0
4,116,108420.655172,108000.0
5,397,133523.347607,133000.0
6,374,161603.034759,160000.0
7,319,207716.423197,200141.0
8,168,274735.535714,269750.0
9,43,367513.023256,345000.0


In [9]:
X = train.drop("SalePrice", axis=1)
y = np.log1p(train["SalePrice"])

In [10]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical:", len(numeric_features))
print("Categorical:", len(categorical_features))

Numerical: 37
Categorical: 43


In [11]:
none_cols = [
    "Alley",
    "MasVnrType",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature"
]

X[none_cols] = X[none_cols].fillna("None")

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [13]:
from sklearn.linear_model import Ridge

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=10))
])

from sklearn.model_selection import KFold, cross_val_score

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="neg_mean_squared_error"
)

rmsle_scores = np.sqrt(-scores)

print("RMSLE:", rmsle_scores)
print("Mean RMSLE:", rmsle_scores.mean())
print("Std:", rmsle_scores.std())

RMSLE: [0.14799682 0.12861026 0.24681501 0.13364802 0.12519921]
Mean RMSLE: 0.15645386398839356
Std: 0.045845140957789986


In [14]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

rf_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="neg_mean_squared_error"
)

rf_rmsle = np.sqrt(-rf_scores)

print("RMSLE:", rf_rmsle)
print("Mean RMSLE:", rf_rmsle.mean())
print("Std:", rf_rmsle.std())

RMSLE: [0.14751578 0.12699829 0.17723317 0.14870575 0.12289608]
Mean RMSLE: 0.14466981268263607
Std: 0.019344965265642132


In [15]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

categorical_transformer_dense = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor_dense = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer_dense, categorical_features)
])

from sklearn.ensemble import GradientBoostingRegressor

gb_model = Pipeline([
    ("preprocessor", preprocessor_dense),
    ("regressor", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb_scores = cross_val_score(
    gb_model,
    X,
    y,
    cv=cv,
    scoring="neg_mean_squared_error"
)

gb_rmsle = np.sqrt(-gb_scores)

print("RMSLE:", gb_rmsle)
print("Mean RMSLE:", gb_rmsle.mean())
print("Std:", gb_rmsle.std())

RMSLE: [0.13353555 0.10984837 0.16685606 0.12995187 0.11185547]
Mean RMSLE: 0.13040946576987
Std: 0.0205199677443586


In [16]:
X_fe = X.copy()

X_fe["TotalSF"] = (
    X_fe["TotalBsmtSF"]
    + X_fe["1stFlrSF"]
    + X_fe["2ndFlrSF"]
)

X_fe["TotalBath"] = (
    X_fe["FullBath"]
    + 0.5 * X_fe["HalfBath"]
    + X_fe["BsmtFullBath"]
    + 0.5 * X_fe["BsmtHalfBath"]
)

X_fe["TotalPorchSF"] = (
    X_fe["OpenPorchSF"]
    + X_fe["3SsnPorch"]
    + X_fe["EnclosedPorch"]
    + X_fe["ScreenPorch"]
    + X_fe["WoodDeckSF"]
)

X_fe["HouseAge"] = X_fe["YrSold"] - X_fe["YearBuilt"]
X_fe["RemodAge"] = X_fe["YrSold"] - X_fe["YearRemodAdd"]

In [17]:
numeric_features_fe = X_fe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_fe = X_fe.select_dtypes(
    include=["object"]
).columns.tolist()

preprocessor_fe = ColumnTransformer([
    ("num", numeric_transformer, numeric_features_fe),
    ("cat", categorical_transformer_dense, categorical_features_fe)
])

In [18]:
gb_model_fe = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("regressor", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb_fe_scores = cross_val_score(
    gb_model_fe,
    X_fe,
    y,
    cv=cv,
    scoring="neg_mean_squared_error"
)

gb_fe_rmsle = np.sqrt(-gb_fe_scores)

print("RMSLE:", gb_fe_rmsle)
print("Mean RMSLE:", gb_fe_rmsle.mean())
print("Std:", gb_fe_rmsle.std())

RMSLE: [0.13489066 0.11046807 0.16354906 0.13148379 0.11602521]
Mean RMSLE: 0.13128335815788697
Std: 0.018548399118237798


In [19]:
X_ms = X.copy()

X_ms["MSSubClass"] = X_ms["MSSubClass"].astype(str)


In [20]:
numeric_features_ms = X_ms.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_ms = X_ms.select_dtypes(
    include=["object"]
).columns.tolist()

In [21]:
preprocessor_ms = ColumnTransformer([
    ("num", numeric_transformer, numeric_features_ms),
    ("cat", categorical_transformer_dense, categorical_features_ms)
])

gb_model_ms = Pipeline([
    ("preprocessor", preprocessor_ms),
    ("regressor", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

ms_scores = cross_val_score(
    gb_model_ms,
    X_ms,
    y,
    cv=cv,
    scoring="neg_mean_squared_error"
)

ms_rmsle = np.sqrt(-ms_scores)

print("RMSLE:", ms_rmsle)
print("Mean RMSLE:", ms_rmsle.mean())
print("Std:", ms_rmsle.std())

RMSLE: [0.1348789  0.10949835 0.17091084 0.12902139 0.10943391]
Mean RMSLE: 0.13074867962897663
Std: 0.0225341402168081


In [22]:
gb_tuned = Pipeline([
    ("preprocessor", preprocessor_dense),
    ("regressor", GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    ))
])

tuned_scores = cross_val_score(
    gb_tuned,
    X,
    y,
    cv=cv,
    scoring="neg_mean_squared_error"
)

tuned_rmsle = np.sqrt(-tuned_scores)

print("RMSLE:", tuned_rmsle)
print("Mean RMSLE:", tuned_rmsle.mean())
print("Std:", tuned_rmsle.std())

RMSLE: [0.13303068 0.1100605  0.17036039 0.13087354 0.11131853]
Mean RMSLE: 0.1311287291424567
Std: 0.021813388527396345


In [23]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor

# ============================================================
# 2. SEPARATE FEATURES AND TARGET
# ============================================================

X = train.drop("SalePrice", axis=1)

# RMSLE works with log-transformed target
y = np.log1p(train["SalePrice"])


# ============================================================
# 3. HANDLE SEMANTIC MISSING VALUES
# ============================================================

# NA in these columns means the feature does not exist.
none_cols = [
    "Alley",
    "MasVnrType",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature"
]

# Apply to both train and test
X[none_cols] = X[none_cols].fillna("None")
test[none_cols] = test[none_cols].fillna("None")


# ============================================================
# 4. IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


# ============================================================
# 5. NUMERIC PREPROCESSING
# ============================================================

numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])


# ============================================================
# 6. CATEGORICAL PREPROCESSING
# ============================================================

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


# ============================================================
# 7. COMBINE PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])


# ============================================================
# 8. CREATE BEST MODEL
# ============================================================

model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "regressor",
        GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
    )
])


# ============================================================
# 9. TRAIN ON ALL TRAINING DATA
# ============================================================

model.fit(X, y)


# ============================================================
# 10. PREDICT TEST SET
# ============================================================

# Predictions are still in log-space
test_pred_log = model.predict(test)


# ============================================================
# 11. CONVERT BACK TO ORIGINAL PRICE
# ============================================================

test_pred = np.expm1(test_pred_log)


# ============================================================
# 12. CREATE KAGGLE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_pred
})


# ============================================================
# 13. SAVE SUBMISSION
# ============================================================

submission.to_csv(
    "submission.csv",
    index=False
)


# ============================================================
# 14. CHECK RESULT
# ============================================================

print(submission.head())
print()
print("Shape:", submission.shape)
print()
print("Saved as submission.csv")

     Id      SalePrice
0  1461  120229.145777
1  1462  154538.061525
2  1463  180123.490271
3  1464  187832.943045
4  1465  188699.721178

Shape: (1459, 2)

Saved as submission.csv
